# E1.8 · Third-party and model supply chain risk

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.7 · Continuous control verification](https://spbreed.github.io/cyber-commons/lessons/E1.7.html)**.

| | |
|---|---|
| Tools used | OWASP AIBOM, Sigstore |

## What this lesson is

**What it covers.** Run a real AIBOM against a vendor model artefact.

**Why a security engineer needs it.** Vendor AI features enabled by default; sub-processor chains you never mapped. The control it builds is: questions that actually discriminate between vendors.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Your model vendor, your hosting, your adapters and your MCP servers are all somebody else's risk decisions, inherited. Diligence questions that produce real answers are specific; the generic questionnaire produces a filing.

> **At CyberTravels.** CyberTravels inherited its model vendor's decisions, its OCR library's, and a third-party MCP server's. R4.

## 2 · The framework

```
   inherited decisions

   model vendor  --> training data, safety posture, retention
   hosting       --> where inference happens, what is logged
   adapters      --> who built them, against which base
   agent tooling --> MCP servers, their tool descriptions

   generic questionnaire -> a filing
   specific question     -> an answer you can act on
```

Third-party risk for AI has the ordinary supply-chain problem plus a question
nobody's assessment form asks:

> **Can this component change without telling us?**

For a library the answer is no — you pin a version. For a hosted model the
answer is usually yes, and it changes the risk rating, because every control you
tested was tested against behaviour the vendor can replace on a Tuesday.

Three artefact classes, with genuinely different maturity:

- **Libraries** — signing, version pinning, download signals. Mature.
- **Model weights or a hosted model** — attestation possible and rare; no
  popularity signal that means anything; version stability is a contractual
  question, not a technical one.
- **Prompt and tool packages (MCP, skills)** — no signing convention, and they
  run with your agent's authority.

Saying which signals are unavailable is part of the assessment, not a gap in it.

## 3 · The procedure, as a skill

The skill scores each AI component on the two properties that make it different — silent change, and running with the agent's authority — then invalidates every control test taken before the model changed, because a test against a different model is evidence about something else.

### The skill — [`skills/grc/third-party-ai-assessment/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/grc/third-party-ai-assessment/SKILL.md)

```yaml
name: third-party-ai-assessment
description: >-
  Assess the AI components of a supply chain for the two properties that make
  them different — silent change, and running with agent authority — and
  invalidate the control tests that predate a model change. Use when a model
  provider changed the model and you need to know what that invalidates, and at
  vendor assessment or renewal.
allowed-tools: Read, Grep, Glob
```

# The vendor changed the model and your control tests expired

Third-party AI differs from ordinary third-party software in two ways. A hosted
model can change underneath you with no change record on your side, which
invalidates every control test taken before it. And a tool package or MCP
connector runs **with your agent's authority**, so its risk is not the
vendor's — it is yours.

## When to use this

Vendor assessment, renewal, and any time a provider announces a model update.

## Procedure

**1 — Enumerate the AI components.** Hosted models, tool packages, MCP servers,
embedded features in products you already bought. The last category is the one
nobody lists.

**2 — Score each on the two properties.** Can it change without telling you, and
does it execute with your agent's authority? Either one alone justifies a higher
tier than the ordinary assessment would give.

**3 — Record the last known model version, with a date.** Without it you cannot
tell whether a control test predates a change, which makes the next step
impossible.

**4 — Invalidate control tests taken before the change.** Not "review" — mark
them unevidenced. A test performed against a different model is not weak
evidence, it is evidence about something else.

**5 — Ask the questions a contract can answer.** Notice period for model change,
whether the version is pinnable, what telemetry you get, and exit. Then record
which ones the vendor declined; that list is the assessment.

## Output contract

```json
{
  "components": [{"name": "str", "kind": "model|tool|mcp|embedded",
                  "silent_change": true, "runs_with_agent_authority": false, "tier": "str"}],
  "versions": [{"component": "str", "version": "str", "as_of": "str", "changed_at": "str|null"}],
  "control_tests": [{"id": "str", "tested_at": "str", "status": "valid|unevidenced", "why": "str"}],
  "contract_questions": [{"question": "str", "answered": false}]
}
```

## Failure modes

- **Assessing the vendor and not the authority.** The connector runs as your
  agent.
- **Keeping a control test that predates a model change.** It evidences the old
  model.
- **Missing embedded AI features.** They arrived with a product you already
  own.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/grc/third-party-ai-assessment/scripts/third_party_ai_assessment.py
SCRIPT = "skills/grc/third-party-ai-assessment/scripts/third_party_ai_assessment.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The hosted model and the MCP tool package both tier high — one for silent change, one for running with agent authority. The silent model change invalidates all three control tests taken before it. The signal table shows libraries with 4 of 4 signals available and hosted models with 0 of 4, and each assessment statement names what was unavailable.

## Your turn

Add "can this change without notifying us?" to your third-party assessment form. For hosted models the answer is usually yes, and it should carry an explicit control-test expiry.

---

**Next → [E1.9 · Model and agent lifecycle governance](https://spbreed.github.io/cyber-commons/lessons/E1.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*